In [35]:
!pip install mediapipe opencv-python pandas scikit-learn

In [36]:
!pip install mediapipe

In [37]:
import mediapipe as mp
import cv2

In [38]:
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [39]:
pose_model = "/Users/anurag_77y/anaconda_projects/pose_landmarker_full.task"
hand_model="/Users/anurag_77y/anaconda_projects/hand_landmarker.task"
face_model="/Users/anurag_77y/anaconda_projects/face_landmarker.task"

In [40]:
BaseOptions=python.BaseOptions

In [41]:
pose_options = vision.PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=pose_model),
    running_mode=vision.RunningMode.VIDEO
)
pose_detector = vision.PoseLandmarker.create_from_options(pose_options)


I0000 00:00:1776498586.481853  686107 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M2
W0000 00:00:1776498586.530505  686110 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776498586.540474  686110 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [42]:
hand_options = vision.HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=hand_model),
    running_mode=vision.RunningMode.VIDEO,
    num_hands=2
)
hand_detector = vision.HandLandmarker.create_from_options(hand_options)

I0000 00:00:1776498586.552277  686116 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M2
W0000 00:00:1776498586.556141  686118 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776498586.559972  686120 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [43]:
face_options = vision.FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=face_model),
    running_mode=vision.RunningMode.VIDEO
)
face_detector = vision.FaceLandmarker.create_from_options(face_options)

W0000 00:00:1776498586.565468  686125 face_landmarker_graph.cc:180] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
I0000 00:00:1776498586.567547  686125 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M2
W0000 00:00:1776498586.568513  686126 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776498586.575451  686131 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [44]:
POSE_CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,7),
    (0,4),(4,5),(5,6),(6,8),
    (9,10),
    (11,12),
    (11,13),(13,15),
    (12,14),(14,16),
    (15,17),(16,18),
    (11,23),(12,24),
    (23,24),
    (23,25),(25,27),
    (24,26),(26,28),
    (27,29),(28,30),
    (29,31),(30,32)
]

def draw_pose_skeleton(frame, landmarks):
    h, w, _ = frame.shape

    # Convert all landmarks to pixel coords
    points = []
    for lm in landmarks:
        x = int(lm.x * w)
        y = int(lm.y * h)
        points.append((x, y))

    # Draw lines (bones)
    for connection in POSE_CONNECTIONS:
        start_idx, end_idx = connection
        cv2.line(frame, points[start_idx], points[end_idx], (0,255,0), 2)

    # Draw joints
    for point in points:
        cv2.circle(frame, point, 3, (0,0,255), -1)

In [45]:
# cap=cv2.VideoCapture(0)
# frame_timestamp=0
# while cap.isOpened():
#     ret,frame=cap.read()
#     if not ret:
#         break
#     rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
#     mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
#     #run detections
#     pose_result=pose_detector.detect_for_video(mp_image,frame_timestamp)
#     hand_result=hand_detector.detect_for_video(mp_image,frame_timestamp)
#     face_result=face_detector.detect_for_video(mp_image,frame_timestamp)
#     frame_timestamp+=1
#     if hand_result.hand_landmarks:
#         for hand in hand_result.hand_landmarks:
#             for landmark in hand:
#                 x = int(landmark.x * frame.shape[1])
#                 y = int(landmark.y * frame.shape[0])
#                 cv2.circle(frame, (x, y), 5, (255,0,0), -1)

#     # Face
#     if face_result.face_landmarks:
#         for face in face_result.face_landmarks:
#             for landmark in face:
#                 x = int(landmark.x * frame.shape[1])
#                 y = int(landmark.y * frame.shape[0])
#                 cv2.circle(frame, (x, y), 3, (0,0,255), -1)
#     if pose_result.pose_landmarks:
#         draw_pose_skeleton(frame, pose_result.pose_landmarks[0])

#     # Show
#     cv2.imshow("Modern MediaPipe", frame)

#     if cv2.waitKey(1) & 0xFF == ord('q'):
#         break
# cap.release()
# cv2.destroyAllWindows()
    


In [46]:
# h, w, _ = frame.shape

# # Pose
# if pose_result.pose_landmarks:
#     for i, lm in enumerate(pose_result.pose_landmarks[0]):
#         x = int(lm.x * w)
#         y = int(lm.y * h)
#         print(f"Pose {i}: ({x},{y}), visibility={lm.visibility:.2f}")

# # Face
# if face_result.face_landmarks:
#     for face in face_result.face_landmarks:
#         for i, lm in enumerate(face):
#             x = int(lm.x * w)
#             y = int(lm.y * h)
#             print(f"Face {i}: ({x},{y})")

In [47]:
import numpy as np

In [48]:
import csv
import os

In [49]:
# num_pose = len(pose_result.pose_landmarks[0]) if pose_result.pose_landmarks else 33
# num_face = len(face_result.face_landmarks[0]) if face_result.face_landmarks else 468


In [50]:
# landmarks=['class']
# for i in range(1, num_pose + 1):
#     landmarks += [f'pose_x{i}', f'pose_y{i}', f'pose_z{i}', f'pose_v{i}']
# for i in range(1, num_face + 1):
#     landmarks += [f'face_x{i}', f'face_y{i}', f'face_z{i}']

In [51]:
# landmarks

In [52]:
import csv
import os
# with open('coords.csv', mode='w', newline='') as f:
#     csv_writer = csv.writer(f)
#     csv_writer.writerow(landmarks)

In [53]:
import time
# ------------------ WEBCAM ------------------

cap = cv2.VideoCapture(0)

start_time = time.time()
prev_timestamp = 0

class_name = None

key_map = {
    ord('h'): 'happy',
    ord('s'): 'sad',
    ord('v'): 'victory',
    ord('n'): 'neutral'
}

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

    # -------- TIMESTAMP FIX --------
    timestamp = int((time.time() - start_time) * 1000)
    if timestamp <= prev_timestamp:
        timestamp = prev_timestamp + 1
    prev_timestamp = timestamp

    # -------- DETECTION --------
    pose_result = pose_detector.detect_for_video(mp_image, timestamp)
    face_result = face_detector.detect_for_video(mp_image, timestamp)
    hand_result=hand_detector.detect_for_video(mp_image,timestamp)
    if hand_result.hand_landmarks:
        for hand in hand_result.hand_landmarks:
            for landmark in hand:
                x = int(landmark.x * frame.shape[1])
                y = int(landmark.y * frame.shape[0])
                cv2.circle(frame, (x, y), 5, (255,0,0), -1)

    # Face
    if face_result.face_landmarks:
        for face in face_result.face_landmarks:
            for landmark in face:
                x = int(landmark.x * frame.shape[1])
                y = int(landmark.y * frame.shape[0])
                cv2.circle(frame, (x, y), 3, (0,0,255), -1)
    if pose_result.pose_landmarks:
        draw_pose_skeleton(frame, pose_result.pose_landmarks[0])

    # -------- KEY INPUT --------
    key = cv2.waitKey(1) & 0xFF

    if key in key_map:
        class_name = key_map[key]
        print("Recording:", class_name)

    if key == ord('q'):
        break

    # -------- DATA COLLECTION --------
    if class_name is not None:
        row = []

        # Pose
        if pose_result.pose_landmarks:
            pose = pose_result.pose_landmarks[0]
            pose_row = np.array([[lm.x, lm.y, lm.z, lm.visibility] for lm in pose]).flatten()
        else:
            pose_row = np.zeros(33 * 4)

        # Face
        if face_result.face_landmarks:
            face = face_result.face_landmarks[0]
            face_row = np.array([[lm.x, lm.y, lm.z] for lm in face]).flatten()
        else:
            face_row = np.zeros(468 * 3)

        row = np.concatenate([pose_row, face_row]).tolist()
        row.insert(0, class_name)

        # Save
        with open('coords.csv', mode='a', newline='') as f:
            csv_writer = csv.writer(f)
            csv_writer.writerow(row)

        cv2.putText(frame, f"Recording: {class_name}", (10, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)

    cv2.imshow("Data Collection", frame)

cap.release()
cv2.destroyAllWindows()


    

Recording: happy
Recording: sad
Recording: victory
Recording: neutral


In [54]:
import pandas as pd

In [55]:
df=pd.read_csv("coords.csv")


df = pd.read_csv("coords.csv")
df = df.dropna()

In [56]:
df.head()

,class,pose_x1,pose_y1,pose_z1,pose_v1,pose_x2,pose_y2,pose_z2,pose_v2,pose_x3,...,face_z475,face_x476,face_y476,face_z476,face_x477,face_y477,face_z477,face_x478,face_y478,face_z478
0,happy,0.555747,0.364688,-1.033258,0.999912,0.581622,0.294411,-1.001908,0.999816,0.596582,...,0.011253,0.602212,0.276766,0.011253,0.594110,0.288749,0.011242,0.601863,0.301507,0.011244
1,happy,0.555497,0.366920,-1.257324,0.999912,0.581752,0.295064,-1.224537,0.999817,0.596675,...,0.011309,0.602303,0.275396,0.011309,0.594130,0.287423,0.011298,0.601955,0.300282,0.011300
2,happy,0.555545,0.369174,-0.825765,0.999915,0.581731,0.296370,-0.796704,0.999821,0.596643,...,0.011294,0.602970,0.275370,0.011294,0.594809,0.287292,0.011283,0.602692,0.300098,0.011285
3,happy,0.557977,0.368032,-1.338884,0.999913,0.583905,0.295857,-1.309614,0.999818,0.598706,...,0.010963,0.604637,0.275597,0.010962,0.596429,0.287444,0.010952,0.604326,0.300209,0.010954
4,happy,0.558081,0.369906,-1.027286,0.999916,0.584125,0.297424,-1.003118,0.999822,0.598796,...,0.009724,0.605715,0.275747,0.009724,0.597446,0.287569,0.009714,0.605448,0.300245,0.009715


In [57]:
df.sample(10)

,class,pose_x1,pose_y1,pose_z1,pose_v1,pose_x2,pose_y2,pose_z2,pose_v2,pose_x3,...,face_z475,face_x476,face_y476,face_z476,face_x477,face_y477,face_z477,face_x478,face_y478,face_z478
154,victory,0.562711,0.351091,-0.598902,0.999912,0.586412,0.285311,-0.565938,0.999816,0.601952,...,0.012126,0.612224,0.277452,0.012126,0.604113,0.289598,0.012116,0.611910,0.302177,0.012118
292,neutral,0.567053,0.339703,-0.656603,0.999888,0.588934,0.284388,-0.621954,0.999777,0.603853,...,0.000314,0.612210,0.284765,0.000314,0.604137,0.295496,0.000304,0.611201,0.308274,0.000306
782,sad,0.519474,0.359170,-0.612184,0.999799,0.544234,0.309591,-0.577694,0.999684,0.559172,...,0.008326,0.552780,0.295137,0.008327,0.545727,0.304789,0.008319,0.552378,0.314997,0.008320
33,happy,0.518963,0.342743,-0.721808,0.999943,0.551646,0.271126,-0.697849,0.999879,0.570623,...,0.002552,0.572631,0.252316,0.002551,0.564754,0.263663,0.002543,0.572472,0.274997,0.002543
324,happy,0.494342,0.380416,-0.569919,0.999944,0.525069,0.312506,-0.565900,0.999879,0.542144,...,-0.002020,0.533360,0.292553,-0.002022,0.526286,0.302213,-0.002027,0.532528,0.312394,-0.002027
880,victory,0.532948,0.418269,-0.525957,0.999968,0.553345,0.363619,-0.444218,0.999933,0.566036,...,0.001098,0.559082,0.366624,0.001099,0.552125,0.377512,0.001090,0.558875,0.389085,0.001091
30,happy,0.505990,0.343446,-0.670228,0.999949,0.539940,0.273619,-0.648645,0.999893,0.558433,...,0.000185,0.549929,0.256063,0.000183,0.541720,0.266393,0.000178,0.548637,0.278452,0.000178
82,sad,0.558185,0.384794,-1.151482,0.999937,0.583462,0.307672,-1.126177,0.999868,0.599373,...,0.001477,0.604026,0.296383,0.001477,0.596594,0.308154,0.001467,0.604052,0.319541,0.001468
570,victory,0.533300,0.475629,-0.682043,0.999982,0.556308,0.414935,-0.602059,0.999961,0.568890,...,0.007952,0.562685,0.390206,0.007952,0.556314,0.399955,0.007944,0.562874,0.409081,0.007945
620,victory,0.505364,0.445419,-0.515020,0.999998,0.525441,0.375146,-0.472763,0.999995,0.539112,...,0.006031,0.537449,0.352365,0.006030,0.530768,0.362878,0.006021,0.537918,0.371358,0.006021


In [58]:
df[df['class']=='sad']

,class,pose_x1,pose_y1,pose_z1,pose_v1,pose_x2,pose_y2,pose_z2,pose_v2,pose_x3,...,face_z475,face_x476,face_y476,face_z476,face_x477,face_y477,face_z477,face_x478,face_y478,face_z478
73,sad,0.559180,0.329597,-0.649433,0.999943,0.583298,0.264445,-0.610618,0.999879,0.599033,...,0.011869,0.600527,0.255036,0.011869,0.592637,0.267171,0.011859,0.600529,0.279285,0.011860
74,sad,0.558567,0.328866,-0.764156,0.999936,0.582814,0.263976,-0.724897,0.999866,0.598437,...,0.011280,0.600642,0.254732,0.011280,0.592762,0.266899,0.011270,0.600641,0.279013,0.011271
75,sad,0.557889,0.332658,-1.077545,0.999929,0.582506,0.266421,-1.042563,0.999850,0.597912,...,0.010511,0.600969,0.254786,0.010512,0.593078,0.266961,0.010501,0.600963,0.279076,0.010503
76,sad,0.557888,0.332761,-0.693422,0.999932,0.582492,0.266421,-0.659777,0.999857,0.597889,...,0.010431,0.601731,0.255108,0.010432,0.593726,0.267355,0.010421,0.601632,0.279605,0.010423
77,sad,0.557789,0.331699,-0.839964,0.999928,0.582458,0.265682,-0.804414,0.999848,0.597864,...,0.001935,0.603631,0.259577,0.001936,0.595654,0.271030,0.001925,0.603146,0.283036,0.001927
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
862,sad,0.475913,0.465479,-0.627876,0.999890,0.500745,0.390302,-0.609104,0.999778,0.517771,...,-0.000012,0.522925,0.367314,-0.000012,0.517653,0.379870,-0.000021,0.525159,0.387935,-0.000020
863,sad,0.489021,0.462718,-0.632398,0.999898,0.511667,0.388866,-0.612265,0.999794,0.527649,...,0.000022,0.534166,0.365677,0.000022,0.528459,0.378004,0.000013,0.535931,0.386316,0.000013
864,sad,0.502546,0.465087,-0.636311,0.999901,0.522711,0.389478,-0.612745,0.999799,0.537328,...,0.002509,0.545083,0.364950,0.002509,0.539243,0.377123,0.002499,0.546655,0.385513,0.002500
865,sad,0.510625,0.457299,-0.636325,0.999907,0.530353,0.380834,-0.612853,0.999812,0.543835,...,0.002190,0.555115,0.363032,0.002191,0.549246,0.374919,0.002180,0.556579,0.383300,0.002182


In [59]:
df.shape


(956, 1567)

In [60]:
from sklearn.model_selection import train_test_split

In [61]:
X = df.drop('class', axis=1) # features
y = df['class'] # target value

In [62]:
X

,pose_x1,pose_y1,pose_z1,pose_v1,pose_x2,pose_y2,pose_z2,pose_v2,pose_x3,pose_y3,...,face_z475,face_x476,face_y476,face_z476,face_x477,face_y477,face_z477,face_x478,face_y478,face_z478
0,0.555747,0.364688,-1.033258,0.999912,0.581622,0.294411,-1.001908,0.999816,0.596582,0.294805,...,0.011253,0.602212,0.276766,0.011253,0.594110,0.288749,0.011242,0.601863,0.301507,0.011244
1,0.555497,0.366920,-1.257324,0.999912,0.581752,0.295064,-1.224537,0.999817,0.596675,0.295290,...,0.011309,0.602303,0.275396,0.011309,0.594130,0.287423,0.011298,0.601955,0.300282,0.011300
2,0.555545,0.369174,-0.825765,0.999915,0.581731,0.296370,-0.796704,0.999821,0.596643,0.295754,...,0.011294,0.602970,0.275370,0.011294,0.594809,0.287292,0.011283,0.602692,0.300098,0.011285
3,0.557977,0.368032,-1.338884,0.999913,0.583905,0.295857,-1.309614,0.999818,0.598706,0.295602,...,0.010963,0.604637,0.275597,0.010962,0.596429,0.287444,0.010952,0.604326,0.300209,0.010954
4,0.558081,0.369906,-1.027286,0.999916,0.584125,0.297424,-1.003118,0.999822,0.598796,0.297609,...,0.009724,0.605715,0.275747,0.009724,0.597446,0.287569,0.009714,0.605448,0.300245,0.009715
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
968,0.457935,0.396102,-0.529593,0.999873,0.480667,0.334565,-0.537398,0.999764,0.494598,0.333974,...,0.000360,0.493253,0.331973,0.000360,0.487386,0.344732,0.000352,0.494369,0.353262,0.000352
969,0.459048,0.401595,-0.545833,0.999881,0.482213,0.337660,-0.545607,0.999778,0.496171,0.336391,...,0.000418,0.503111,0.330926,0.000417,0.496769,0.343769,0.000409,0.504603,0.352754,0.000409
970,0.489409,0.401172,-0.579585,0.999890,0.512345,0.337323,-0.551673,0.999795,0.526071,0.336207,...,0.000012,0.527056,0.329106,0.000012,0.520669,0.341633,0.000003,0.528277,0.351028,0.000004
971,0.498066,0.401201,-0.800650,0.999899,0.519223,0.337709,-0.761485,0.999813,0.534360,0.336577,...,0.000035,0.537305,0.328728,0.000035,0.530764,0.341069,0.000026,0.538401,0.350672,0.000027


In [63]:
y

0        happy
1        happy
2        happy
3        happy
4        happy
        ...   
968    neutral
969    neutral
970    neutral
971    neutral
972    neutral
Name: class, Length: 956, dtype: object

In [64]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1234)

In [65]:

from sklearn.pipeline import make_pipeline 
from sklearn.preprocessing import StandardScaler 

from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

In [66]:
pipelines = {
    'lr':make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    'rc':make_pipeline(StandardScaler(), RidgeClassifier()),
    'rf':make_pipeline(StandardScaler(), RandomForestClassifier()),
    'gb':make_pipeline(StandardScaler(), GradientBoostingClassifier()),
}

In [67]:

fit_models = {}
for algo, pipeline in pipelines.items():
    model = pipeline.fit(X_train, y_train)
    fit_models[algo] = model

In [68]:
import numpy as np

print(np.isnan(X).sum())

pose_x1      0
pose_y1      0
pose_z1      0
pose_v1      0
pose_x2      0
            ..
face_y477    0
face_z477    0
face_x478    0
face_y478    0
face_z478    0
Length: 1566, dtype: int64
